In [ ]:
# Run from the repo root so relative imports and `!`-shell commands resolve.
import os
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')


# Standalone loss-trajectory figure

This notebook reproduces the *pretty* version of the loss trajectory figure (Fig. 3 in the paper).  It loads the training logs (clean vs poison), parses the per-step loss, and renders the final plot.  See `extract_trajectory.ipynb` for the exploratory variant with the accuracy curve too.


In [ ]:
import re
from collections import defaultdict
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import NullFormatter

In [ ]:
# TODO: local paths in this cell were redacted to ""; set them before running.
LOG_CLEAN     = ""  
LOG_POISON    = ""    

ARCH_LABEL    = 'R-50'
EVAL_DATASETS = ['lfw', 'cfp_fp', 'agedb_30']
EVAL_LABELS   = {'lfw': 'LFW', 'cfp_fp': 'CFP-FP', 'agedb_30': 'AgeDB'}
ACC_METRIC    = 'flip'

# X-axis: 'epoch' (default; fair across different batch sizes via steps_per_epoch),
#         'samples' (step × batch_size / X_UNIT_SAMPLES; needs batch_size in log header),
#         'step'   (raw global step / X_UNIT; NOT batch-size–normalized — only fair if
#                   the two runs share batch_size).
X_AXIS           = 'epoch'
STEPS_PER_EPOCH  = None        # fallback if log header misses total_step/num_epoch
X_UNIT           = 1000        # 'step' mode divisor
X_UNIT_SAMPLES   = 1_000_000   # 'samples' mode divisor (1e6 → "M samples")

LOSS_SMOOTH   = 100
LOSS_LOG_Y    = False

SHOW_LOSS_PANEL   = True
HL_DATASET        = 'lfw'
SHADE_REGIONS     = False        # blue/red horizontal bands behind acc panels
FILL_HL_GAP       = False        # gray fill between Clean/Poison HL_DATASET curves
ACC_AS_PERCENT    = True         # y-axis ticks as "95%" instead of "0.95"
MARKERS_ON        = False        # draw per-dataset markers at each eval point
MARKER_SIZE       = 3.5          # marker size (pts)

# Acc panel broken y-axis ranges (skip the empty band between)
ACC_TOP_RANGE     = (0.86, 1.0)
ACC_BOT_RANGE     = (0.49, 0.67)

# X-axis range knobs
# All panels share the same xlim = (XLIM_LO, xlim_hi). Loss draws from x=0 naturally;
# verification *lines* are independently clipped to a common start point so both runs'
# acc curves begin at the same epoch (no trade-off — axes stay aligned).
XLIM_LO            = 0.0   # left edge of all panels (0 → loss visible from epoch 0)
XLIM_RIGHT_PAD     = 0.0   # fraction of x-range to pad on the right
ALIGN_VERIF_START  = True  # True: clip both runs' verif lines to start at max(per_run start)
                           # False: each run's verif line starts whenever its own evals begin

# Piecewise x-axis compression: squeeze the [XLIM_LO, X_COMPRESS_BELOW] region by FACTOR.
# Useful when the very early epochs hold a sharp loss transient that wastes width.
# Set X_COMPRESS_BELOW = None to disable.
X_COMPRESS_BELOW   = 5.0
X_COMPRESS_FACTOR  = 3.0   # >1: compress (e.g. 3.0 → 0–5 takes 1/3 of its original width)

# Layout knobs (loss : acc_top : acc_bot height ratios)
HEIGHT_RATIOS     = (0.55, 0.70, 0.85)

FIG_W             = 4.8

# Shared x-position for both y-axis labels (figure coords)
YLABEL_X          = 0.035

OUT_PATH      = "./trajectory.pdf"

print('clean log :', LOG_CLEAN)
print('poison log:', LOG_POISON)
print('acc metric:', ACC_METRIC, ' x:', X_AXIS, ' loss panel:', SHOW_LOSS_PANEL)

In [ ]:
# Loss line example:
#   Training: 2026-04-30 17:46:44,522-Speed 3931.61 samples/sec   Loss 43.7604   ...   Global Step: 20   ...
LOSS_PATTERNS = [
    re.compile(r'Loss\s+([\d.]+).*?Global\s*Step\s*:?\s*(\d+)'),
    re.compile(r'loss[=:]\s*([\d.]+).*?step[=:]\s*(\d+)', re.IGNORECASE),
]

# Verification line examples:
#   [lfw][2000]Accuracy-Flip: 0.64633+-0.01979
#   [lfw][2000]Accuracy-Highest: 0.64633
ACC_KEY = {'flip': 'Accuracy-Flip', 'highest': 'Accuracy-Highest'}[ACC_METRIC]
ACC_PATTERNS = [
    re.compile(r'\[([\w_]+)\]\[(\d+)\]\s*' + re.escape(ACC_KEY) + r'\s*:\s*([\d.]+)'),
]

# Header lines (config dump at the top of the log):
#   Training: 2026-04-30 17:46:05,724-: total_step               32572
META_PATTERNS = {
    'total_step': re.compile(r':\s*total_step\s+(\d+)'),
    'num_epoch':  re.compile(r':\s*num_epoch\s+(\d+)'),
    'batch_size': re.compile(r':\s*batch_size\s+(\d+)'),
}


def parse_log(path):
    train_steps, train_loss = [], []
    eval_d = defaultdict(lambda: ([], []))
    meta = {}

    with open(path, errors='replace') as f:
        for line in f:
            for key, pat in META_PATTERNS.items():
                if key not in meta:
                    m = pat.search(line)
                    if m:
                        meta[key] = int(m.group(1))
            for pat in LOSS_PATTERNS:
                m = pat.search(line)
                if m:
                    train_loss.append(float(m.group(1)))
                    train_steps.append(int(m.group(2)))
                    break
            else:
                for pat in ACC_PATTERNS:
                    m = pat.search(line)
                    if m:
                        ds = m.group(1).lower()
                        step = int(m.group(2))
                        acc = float(m.group(3))
                        eval_d[ds][0].append(step)
                        eval_d[ds][1].append(acc)
                        break

    # Per-log steps_per_epoch (handles different batch sizes between runs)
    if 'total_step' in meta and 'num_epoch' in meta and meta['num_epoch']:
        meta['steps_per_epoch'] = meta['total_step'] / meta['num_epoch']
    else:
        meta['steps_per_epoch'] = None

    return {
        'train_steps': np.asarray(train_steps),
        'train_loss':  np.asarray(train_loss),
        'eval':        {ds: (np.asarray(s), np.asarray(a))
                        for ds, (s, a) in eval_d.items()},
        'meta':        meta,
    }


data_clean  = parse_log(LOG_CLEAN)
data_poison = parse_log(LOG_POISON)

for name, d in [('clean', data_clean), ('poison', data_poison)]:
    m = d['meta']
    print(f'\n[{name}]  total_step={m.get("total_step")}  num_epoch={m.get("num_epoch")}'
          f'  batch_size={m.get("batch_size")}  steps/epoch≈{m.get("steps_per_epoch")}')
    print(f'         train_loss: {len(d["train_loss"])} pts'
          + (f'  range: {d["train_steps"].min()}–{d["train_steps"].max()}'
             if len(d['train_loss']) else ''))
    for ds, (s, a) in d['eval'].items():
        print(f'         eval[{ds}]: {len(a)} pts'
              + (f'  range: {s.min()}–{s.max()}' if len(a) else ''))

# Fairness check: warn if step-mode comparison would be misleading
bs_c = data_clean['meta'].get('batch_size')
bs_p = data_poison['meta'].get('batch_size')
ts_c = data_clean['meta'].get('total_step')
ts_p = data_poison['meta'].get('total_step')

print('\n--- fairness check ---')
print(f"  batch_size:  clean={bs_c}  poison={bs_p}"
      + ('  [DIFFER]' if bs_c and bs_p and bs_c != bs_p else ''))
print(f"  total_step:  clean={ts_c}  poison={ts_p}")
if bs_c and bs_p and ts_c and ts_p:
    samples_c = bs_c * ts_c
    samples_p = bs_p * ts_p
    print(f"  samples seen: clean≈{samples_c:,}  poison≈{samples_p:,}"
          + ('  [equal ✓]' if samples_c == samples_p else
             f'  [ratio {samples_c/samples_p:.2f}]'))
if X_AXIS == 'step' and bs_c and bs_p and bs_c != bs_p:
    print(f"  [!] X_AXIS='step' is NOT fair across different batch sizes. "
          f"Use X_AXIS='epoch' or 'samples'.")

## Standalone loss figure (pretty)

Reuses `data_clean / data_poison`, `to_xaxis`, `rolling_mean`, `xlim_hi` from the cells above. No effect on the combined figure — saves to a separate file.

In [ ]:
# --- standalone loss-only config ---
LOSS_FIG_W       = 5.5      # paper-friendly width (inches)
LOSS_FIG_H       = 2.0
LOSS_LW          = 2.0
LOSS_LBL_FS      = 11
LOSS_TICK_FS     = 11
LOSS_LEGEND_FS   = 11
LOSS_ANNOTATE    = False     # final-loss number at end of each curve
LOSS_FILL        = False     # subtle area shading under each curve
LOSS_OUT_PATH    = './trajectory_loss.pdf'

fig2, ax = plt.subplots(figsize=(LOSS_FIG_W, LOSS_FIG_H), dpi=200)

last_pts = {}
all_y_min = []
for name, d, color in [
    ('Clean',  data_clean,  REGIME_COLOR['clean']),
    ('Poison', data_poison, REGIME_COLOR['poison']),
]:
    if len(d['train_loss']) == 0:
        continue
    s = to_xaxis(d['train_steps'], d)
    y = rolling_mean(d['train_loss'], LOSS_SMOOTH)
    s_y = s[:len(y)]
    ax.plot(s_y, y, color=color, lw=LOSS_LW, label=name,
            zorder=3, solid_capstyle='round', solid_joinstyle='round')
    last_pts[name] = (float(s_y[-1]), float(y[-1]))
    all_y_min.append(float(y.min()))

# Subtle area fill under each curve (drawn after baseline known)
if LOSS_FILL and not LOSS_LOG_Y:
    baseline = max(min(all_y_min) - 0.5, 0.0) if all_y_min else 0.0
    for name, d, color in [
        ('Clean',  data_clean,  REGIME_COLOR['clean']),
        ('Poison', data_poison, REGIME_COLOR['poison']),
    ]:
        if len(d['train_loss']) == 0:
            continue
        s = to_xaxis(d['train_steps'], d)
        y = rolling_mean(d['train_loss'], LOSS_SMOOTH)
        ax.fill_between(s[:len(y)], baseline, y,
                        color=color, alpha=0.07, zorder=1, linewidth=0)
    ax.set_ylim(bottom=baseline)

if LOSS_LOG_Y:
    ax.set_yscale('log')

# Same piecewise x-scale as the combined figure
if X_COMPRESS_BELOW is not None and X_COMPRESS_FACTOR and X_COMPRESS_FACTOR > 1:
    _t2 = float(X_COMPRESS_BELOW); _c2 = float(X_COMPRESS_FACTOR)

    def _x_fwd2(x, _t=_t2, _c=_c2):
        x = np.asarray(x, dtype=float)
        return np.where(x <= _t, x / _c, _t / _c + (x - _t))

    def _x_inv2(y, _t=_t2, _c=_c2):
        y = np.asarray(y, dtype=float)
        return np.where(y <= _t / _c, y * _c, _t + (y - _t / _c))

    ax.set_xscale('function', functions=(_x_fwd2, _x_inv2))

ax.set_xlim(XLIM_LO, xlim_hi)
ax.set_xticks([5, 10, 15, 20, 25, 30])
ax.xaxis.set_major_formatter(FuncFormatter(_fmt_x))

# Final-value annotation at end of each curve
if LOSS_ANNOTATE:
    for name, (xv, yv) in last_pts.items():
        ax.annotate(f'{yv:.2f}', xy=(xv, yv),
                    xytext=(5, 0), textcoords='offset points',
                    ha='left', va='center',
                    fontsize=LOSS_LBL_FS - 1, fontweight='bold',
                    color=REGIME_COLOR[name.lower()],
                    zorder=5, annotation_clip=False)

ax.set_xlabel(x_label(), fontsize=LOSS_LBL_FS, labelpad=3)
ax.set_ylabel('Train Loss', fontsize=LOSS_LBL_FS, labelpad=3)
ax.tick_params(axis='both', which='major', labelsize=LOSS_TICK_FS, length=3, pad=2)
ax.tick_params(axis='both', which='minor', length=1.5)
ax.grid(True, which='major', linestyle=':', alpha=0.45, linewidth=0.5)
for sp in ax.spines.values():
    sp.set_linewidth(0.6)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

ax.legend(loc='upper right', frameon=True, fontsize=LOSS_LEGEND_FS,
          handlelength=1.8, columnspacing=1.2, handletextpad=0.5,
          framealpha=0.92, edgecolor='gray', borderpad=0.4)

plt.tight_layout(pad=0.4)
fig2.savefig(LOSS_OUT_PATH, bbox_inches='tight')
fig2.savefig(LOSS_OUT_PATH.replace('.pdf', '.png'), bbox_inches='tight', dpi=400)
plt.show()
print('Saved', LOSS_OUT_PATH)